# 05 — Provider API Comparison — Practical Examples

**Covers**: OpenAI, Anthropic, Gemini, Groq APIs side-by-side, LiteLLM unified interface, response normalization

In [ ]:
import os, json, time
from openai import OpenAI
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client_oai = OpenAI()
MODEL = 'gpt-4o-mini'
TEST_PROMPT = 'In 2 sentences, explain what a transformer neural network is.'
print('✅ OpenAI ready')

## 1. OpenAI API — The Standard Pattern

In [ ]:
# OpenAI: system in messages list, max_tokens optional
t0 = time.perf_counter()
r_oai = client_oai.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'You are a concise AI educator.'},
        {'role': 'user',   'content': TEST_PROMPT}
    ],
    max_tokens=100, temperature=0.7
)
elapsed = (time.perf_counter() - t0) * 1000

print('=== OpenAI response ===')
print(f'Content:        {r_oai.choices[0].message.content}')
print(f'Stop reason:    {r_oai.choices[0].finish_reason}')
print(f'Input tokens:   {r_oai.usage.prompt_tokens}')
print(f'Output tokens:  {r_oai.usage.completion_tokens}')
print(f'Total tokens:   {r_oai.usage.total_tokens}')
print(f'Latency:        {elapsed:.0f}ms')

## 2. Anthropic Claude API — Key Differences

In [ ]:
try:
    import anthropic
    ac = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY', ''))
    
    t0 = time.perf_counter()
    r_claude = ac.messages.create(
        model='claude-3-haiku-20240307',
        system='You are a concise AI educator.',  # ← SEPARATE parameter, not in messages
        messages=[{'role': 'user', 'content': TEST_PROMPT}],
        max_tokens=100,                           # ← REQUIRED (no default)
        temperature=0.7
    )
    elapsed = (time.perf_counter() - t0) * 1000
    
    print('=== Anthropic Claude response ===')
    print(f'Content:        {r_claude.content[0].text}')  # ← .content[0].text not .choices
    print(f'Stop reason:    {r_claude.stop_reason}')       # ← .stop_reason not .finish_reason
    print(f'Input tokens:   {r_claude.usage.input_tokens}')# ← .input_tokens not .prompt_tokens 
    print(f'Output tokens:  {r_claude.usage.output_tokens}')
    print(f'Latency:        {elapsed:.0f}ms')
except ImportError:
    print('📌 pip install anthropic')
    print('   Differences: system=separate param, max_tokens=required, .content[0].text, .input_tokens')
except Exception as e:
    print(f'Check ANTHROPIC_API_KEY: {type(e).__name__}')

## 3. Google Gemini API — model-level System Instruction

In [ ]:
try:
    import google.generativeai as genai
    genai.configure(api_key=os.getenv('GOOGLE_API_KEY', ''))
    
    # Gemini: system_instruction at MODEL level, not message level
    gm = genai.GenerativeModel(
        model_name='gemini-1.5-flash',
        system_instruction='You are a concise AI educator.'  # ← At MODEL level
    )
    
    t0 = time.perf_counter()
    r_gem = gm.generate_content(
        TEST_PROMPT,
        generation_config={'max_output_tokens': 100, 'temperature': 0.7}
    )
    elapsed = (time.perf_counter() - t0) * 1000
    
    print('=== Google Gemini response ===')
    print(f'Content:   {r_gem.text}')         # ← .text directly
    print(f'Latency:   {elapsed:.0f}ms')
    # Note: usage stats available via r_gem.usage_metadata
except ImportError:
    print('📌 pip install google-generativeai')
    print('   Differences: GenerativeModel(system_instruction=...), .text directly, JSON via response_mime_type')
except Exception as e:
    print(f'Check GOOGLE_API_KEY: {type(e).__name__}')

## 4. LiteLLM — One API for All Providers

In [ ]:
try:
    import litellm
    litellm.set_verbose = False
    
    models = [
        'gpt-4o-mini',
        # 'claude-3-haiku-20240307',      # needs ANTHROPIC_API_KEY
        # 'gemini/gemini-1.5-flash',       # needs GOOGLE_API_KEY
    ]
    
    print('=== LiteLLM unified interface ===')
    for model_id in models:
        try:
            t0 = time.perf_counter()
            r = litellm.completion(
                model=model_id,
                messages=[
                    {'role': 'system', 'content': 'You are a concise AI educator.'},
                    {'role': 'user',   'content': TEST_PROMPT}
                ],
                max_tokens=100
            )
            elapsed = (time.perf_counter() - t0) * 1000
            cost = litellm.completion_cost(r)
            
            # LiteLLM normalizes to OpenAI response format
            content = r.choices[0].message.content   # ← Same format for all providers!
            print(f'\n[{model_id}] ({elapsed:.0f}ms, ${cost:.6f})')
            print(f'  {content[:100]}')
        except Exception as e:
            print(f'[{model_id}] {type(e).__name__}')
except ImportError:
    print('📌 pip install litellm')

## 5. Response Normalizer — Unified Access Layer

In [ ]:
from dataclasses import dataclass

@dataclass
class NormalizedResponse:
    content: str
    input_tokens: int
    output_tokens: int
    finish_reason: str
    latency_ms: float
    provider: str
    model: str

def call_openai(prompt: str, model: str = MODEL) -> NormalizedResponse:
    t0 = time.perf_counter()
    r = client_oai.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=100
    )
    return NormalizedResponse(
        content=r.choices[0].message.content,
        input_tokens=r.usage.prompt_tokens,
        output_tokens=r.usage.completion_tokens,
        finish_reason=r.choices[0].finish_reason,
        latency_ms=(time.perf_counter() - t0) * 1000,
        provider='openai', model=model
    )

# Use the normalized layer
response = call_openai(TEST_PROMPT)
print(f'Provider: {response.provider} | Model: {response.model}')
print(f'Latency: {response.latency_ms:.0f}ms | Tokens: {response.input_tokens}→{response.output_tokens}')
print(f'Content: {response.content[:150]}')

## 📌 Summary

| | OpenAI | Anthropic | Gemini | LiteLLM |
|---|---|---|---|---|
| System | In messages | Separate param | Model-level | In messages |
| max_tokens | Optional | **Required** | Optional | Provider-specific |
| Content access | `.choices[0].message.content` | `.content[0].text` | `.text` | `.choices[0].message.content` |
| Tokens | `.prompt_tokens` | `.input_tokens` | `.usage_metadata` | Normalized |
| Best for | Standard default | Claude quality | Large context | Multi-provider |

- **Use LiteLLM** for provider-agnostic code that can switch providers with one line
- **NormalizedResponse dataclass** to abstract provider differences in your codebase
- **Always log token counts** — field names differ but data is critical for cost tracking